# ARM97 Observation, Baseline, Stitched Baseline, and Experiment Comparison

This notebook compares ARM97 observation, the continuous baseline, the stitched-method baseline, and an optional batch of experiment members. By default it points to the current `arm97_qmc14x5_stitched_seed20260602` stitched outputs and the stitched baseline generated under `arm97_stitched_baseline_seed20260605`.

- Upper panel: observation, continuous baseline, stitched baseline, and optional experiment members.
- Lower panel: each model curve minus observation.
- Variables are limited to explicit ARM97 observation mappings from `notebooks/observed_variable_pairs.csv` by default.


In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from datetime import timedelta
import math
import os
from pathlib import Path
import re

def find_repo_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "e3sm_scm_run_scripts_baseline").exists():
            return candidate
    return Path("/Users/yunlong/Workshop/SCM-UQ-Workflow")

ROOT = Path(os.environ.get("SCM_UQ_WORKFLOW_ROOT", find_repo_root())).resolve()
os.environ.setdefault("MPLCONFIGDIR", str(ROOT / ".local_cache/matplotlib-cache"))
os.environ.setdefault("XDG_CACHE_HOME", str(ROOT / ".local_cache"))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from netCDF4 import Dataset, num2date
import plotly.graph_objects as go
from plotly.subplots import make_subplots

ROOT


## Configuration

For a different experiment, edit `EXPERIMENT_DIR`, `FILE_GLOB`, and `RUN_ID_REGEX`. Set `EXPERIMENT_ENABLED = False` to compare only observation, continuous baseline, and stitched baseline.

In [ ]:
DEFAULT_BASELINE = (
    ROOT
    / "e3sm_scm_run_scripts_baseline/baseline-output/scm_ARM97_baseline/run"
    / "case_scripts.eam.h0.1997-06-19-84585.nc"
)
DEFAULT_STITCHED_BASELINE = (
    ROOT
    / "arm97_experiments_0602/arm97_stitched_baseline_seed20260605/mac/stitched"
    / "mac_ARM97_arm97_stitched_baseline_seed20260605_000_stitched_26day.nc"
)
DEFAULT_OBSERVATION = ROOT / "e3sm_scm_run_scripts_baseline/ARM97_iopfile_4scam.nc"
DEFAULT_OBS_MAP = ROOT / "notebooks/observed_variable_pairs.csv"
DEFAULT_EXPERIMENT_DIR = (
    ROOT
    / "arm97_experiments_0602/arm97_qmc14x5_stitched_seed20260602/mac/stitched"
)

BASELINE = Path(os.environ.get("SCM_BASELINE_HISTORY_FILE", DEFAULT_BASELINE)).expanduser().resolve()
STITCHED_BASELINE = Path(os.environ.get("SCM_STITCHED_BASELINE_FILE", DEFAULT_STITCHED_BASELINE)).expanduser().resolve()
OBSERVATION = Path(os.environ.get("ARM97_IOP_FILE", DEFAULT_OBSERVATION)).expanduser().resolve()
OBS_MAP = Path(os.environ.get("ARM97_OBS_MAP", DEFAULT_OBS_MAP)).expanduser().resolve()

EXPERIMENT_ENABLED = True
EXPERIMENT_DIR = Path(os.environ.get("ARM97_EXPERIMENT_DIR", DEFAULT_EXPERIMENT_DIR)).expanduser().resolve()
FILE_GLOB = "mac_ARM97_qmc14x5_*_stitched_26day.nc"
RUN_ID_REGEX = r"mac_ARM97_qmc14x5_(\d{3})_stitched_26day\.nc$"
CASE_LABEL = "arm97_qmc14x5_stitched_seed20260602"

BATCH_SIZE = 10
BATCH_INDEX = 0
SELECTED_RUN_IDS: list[int] | None = None

# Keep None for all mapped observation variables, or use e.g. ["TREFHT", "TMQ", "CLDTOT"].
INCLUDE_VARIABLES: list[str] | None = None

OUT_DIR = Path(
    os.environ.get(
        "ARM97_OBS_BASELINE_STITCHED_EXPERIMENT_OUT_DIR",
        EXPERIMENT_DIR / "observation_baseline_stitched_experiment_figures",
    )
).expanduser().resolve()

for label, path in {
    "baseline": BASELINE,
    "stitched baseline": STITCHED_BASELINE,
    "observation": OBSERVATION,
    "observation map": OBS_MAP,
}.items():
    assert path.exists(), f"missing {label}: {path}"
if EXPERIMENT_ENABLED:
    assert EXPERIMENT_DIR.exists(), EXPERIMENT_DIR

print("baseline:", BASELINE)
print("stitched baseline:", STITCHED_BASELINE)
print("observation:", OBSERVATION)
print("observation map:", OBS_MAP)
print("experiment enabled:", EXPERIMENT_ENABLED)
print("experiment dir:", EXPERIMENT_DIR if EXPERIMENT_ENABLED else "disabled")
print("output dir:", OUT_DIR)


## Observation Mappings

In [ ]:
@dataclass(frozen=True)
class VarSpec:
    model: str
    obs_terms: tuple[str, ...]
    units: str
    scale_obs: float = 1.0
    obs_offset: float = 0.0
    description: str = ""


def load_obs_specs(path: Path = OBS_MAP) -> dict[str, VarSpec]:
    df = pd.read_csv(path)
    specs: dict[str, VarSpec] = {}
    for row in df.itertuples(index=False):
        terms = tuple(
            term.strip()
            for term in str(row.observation_source_variables).replace(";", ",").split(",")
            if term.strip()
        )
        specs[str(row.model_variable)] = VarSpec(
            model=str(row.model_variable),
            obs_terms=terms,
            units="" if pd.isna(row.units_after_conversion) else str(row.units_after_conversion),
            scale_obs=float(row.scale_obs),
            obs_offset=float(row.obs_offset),
            description="" if pd.isna(row.description) else str(row.description),
        )
    return specs


OBS_SPECS = load_obs_specs()
pd.DataFrame([spec.__dict__ for spec in OBS_SPECS.values()]).head(30)


## Experiment Files

In [ ]:
def parse_run_id(path: Path, regex: str = RUN_ID_REGEX):
    match = re.search(regex, path.name)
    if not match:
        return None
    value = match.group(1)
    try:
        return int(value)
    except ValueError:
        return value


def sort_key(value):
    return (0, value) if isinstance(value, int) else (1, str(value))


def run_label(value) -> str:
    return f"{value:03d}" if isinstance(value, int) else str(value)


def discover_experiment_files() -> pd.DataFrame:
    if not EXPERIMENT_ENABLED:
        return pd.DataFrame(columns=["run_id", "path"])
    rows = []
    for index, path in enumerate(sorted(EXPERIMENT_DIR.glob(FILE_GLOB))):
        run_id = parse_run_id(path)
        rows.append({"run_id": index if run_id is None else run_id, "path": path})
    df = pd.DataFrame(rows)
    if df.empty:
        raise FileNotFoundError(f"No files matched {FILE_GLOB!r} in {EXPERIMENT_DIR}")
    return df.sort_values("run_id", key=lambda s: s.map(sort_key)).reset_index(drop=True)


def select_batch(files: pd.DataFrame) -> pd.DataFrame:
    if not EXPERIMENT_ENABLED:
        return files
    if SELECTED_RUN_IDS is not None:
        selected = files[files["run_id"].isin(SELECTED_RUN_IDS)].copy()
        missing = sorted(set(SELECTED_RUN_IDS) - set(selected["run_id"]), key=sort_key)
        if missing:
            raise ValueError(f"Missing requested run ids: {missing}")
        return selected.sort_values("run_id", key=lambda s: s.map(sort_key)).reset_index(drop=True)
    start = BATCH_INDEX * BATCH_SIZE
    selected = files.iloc[start : start + BATCH_SIZE].copy()
    if selected.empty:
        n_batches = math.ceil(len(files) / BATCH_SIZE)
        raise ValueError(f"BATCH_INDEX {BATCH_INDEX} is empty; valid range is 0..{n_batches - 1}")
    return selected.reset_index(drop=True)


ALL_FILES = discover_experiment_files()
BATCH_FILES = select_batch(ALL_FILES)
N_BATCHES = math.ceil(len(ALL_FILES) / BATCH_SIZE) if EXPERIMENT_ENABLED else 0

if EXPERIMENT_ENABLED:
    print(f"found {len(ALL_FILES)} experiment files")
    print(f"batch {BATCH_INDEX + 1}/{N_BATCHES}:", [run_label(x) for x in BATCH_FILES["run_id"]])
BATCH_FILES


## Load Comparison Data

In [ ]:
def as_series(var):
    data = np.ma.asarray(var[:], dtype=np.float64)
    if data.ndim == 1:
        return data
    return np.ma.mean(data, axis=tuple(range(1, data.ndim)))


def filled(arr):
    return np.asarray(np.ma.asarray(arr, dtype=np.float64).filled(np.nan), dtype=np.float64)


def load_time_axis(ds):
    t = ds.variables["time"]
    days = np.asarray(t[:], dtype=np.float64)
    dates = np.array(num2date(days, t.units, getattr(t, "calendar", "standard"), only_use_cftime_datetimes=False))
    return days, dates


def interp_reference(source_days, source_values, target_days):
    finite = np.isfinite(source_values)
    if finite.sum() < 2:
        return np.full_like(target_days, np.nan, dtype=np.float64)
    return np.interp(target_days, source_days[finite], source_values[finite], left=np.nan, right=np.nan)


def score(model_values, reference_values):
    finite = np.isfinite(model_values) & np.isfinite(reference_values)
    if not finite.any():
        return dict(n=0, bias=np.nan, mae=np.nan, rmse=np.nan, max_abs=np.nan)
    diff = model_values[finite] - reference_values[finite]
    return dict(
        n=int(diff.size),
        bias=float(np.mean(diff)),
        mae=float(np.mean(np.abs(diff))),
        rmse=float(np.sqrt(np.mean(diff * diff))),
        max_abs=float(np.max(np.abs(diff))),
    )


def obs_available(obs, spec: VarSpec) -> bool:
    return bool(spec.obs_terms) and all(term in obs.variables for term in spec.obs_terms)


def obs_series(obs, spec: VarSpec):
    values = filled(as_series(obs.variables[spec.obs_terms[0]]))
    for term in spec.obs_terms[1:]:
        values = values - filled(as_series(obs.variables[term]))
    return values * spec.scale_obs + spec.obs_offset


def discover_variable_specs() -> list[VarSpec]:
    include = set(INCLUDE_VARIABLES) if INCLUDE_VARIABLES is not None else None
    specs = []
    sample_path = BATCH_FILES.iloc[0]["path"] if EXPERIMENT_ENABLED and not BATCH_FILES.empty else STITCHED_BASELINE
    with Dataset(BASELINE) as baseline, Dataset(STITCHED_BASELINE) as stitched, Dataset(sample_path) as sample, Dataset(OBSERVATION) as obs:
        for name, spec in OBS_SPECS.items():
            if include is not None and name not in include:
                continue
            if name not in baseline.variables or name not in stitched.variables or not obs_available(obs, spec):
                continue
            if EXPERIMENT_ENABLED and name not in sample.variables:
                continue
            specs.append(spec)
    if not specs:
        raise ValueError("No mapped observation variables are available for this comparison")
    return specs


def load_comparison_batch(specs=None, batch_files=BATCH_FILES):
    specs = discover_variable_specs() if specs is None else list(specs)
    data = {}
    rows = []

    with Dataset(BASELINE) as baseline, Dataset(STITCHED_BASELINE) as stitched_baseline, Dataset(OBSERVATION) as obs:
        baseline_days, baseline_dates = load_time_axis(baseline)
        stitched_days, stitched_dates = load_time_axis(stitched_baseline)
        obs_days = (np.asarray(obs.variables["tsec"][:], dtype=np.float64) - float(obs.variables["tsec"][0])) / 86400.0
        origin = baseline_dates[0] - timedelta(days=float(baseline_days[0]))
        obs_dates = np.array([origin + timedelta(days=float(x)) for x in obs_days])

        for spec in specs:
            obs_values = obs_series(obs, spec)
            baseline_values = filled(as_series(baseline.variables[spec.model]))
            stitched_values = filled(as_series(stitched_baseline.variables[spec.model]))
            obs_at_baseline = interp_reference(obs_days, obs_values, baseline_days)
            obs_at_stitched = interp_reference(obs_days, obs_values, stitched_days)

            references = [
                {
                    "label": "continuous baseline",
                    "days": baseline_days,
                    "dates": baseline_dates,
                    "values": baseline_values,
                    "obs_at_model": obs_at_baseline,
                    "diff_obs": baseline_values - obs_at_baseline,
                    "color": "#145DA0",
                    "width": 2.2,
                },
                {
                    "label": "stitched baseline",
                    "days": stitched_days,
                    "dates": stitched_dates,
                    "values": stitched_values,
                    "obs_at_model": obs_at_stitched,
                    "diff_obs": stitched_values - obs_at_stitched,
                    "color": "#C2410C",
                    "width": 2.2,
                },
            ]

            members = []
            if EXPERIMENT_ENABLED:
                for row in batch_files.itertuples(index=False):
                    with Dataset(row.path) as exp:
                        if spec.model not in exp.variables:
                            continue
                        days, dates = load_time_axis(exp)
                        values = filled(as_series(exp.variables[spec.model]))
                        obs_at_model = interp_reference(obs_days, obs_values, days)
                        members.append(
                            {
                                "run_id": row.run_id,
                                "label": f"experiment {run_label(row.run_id)}",
                                "path": Path(row.path),
                                "days": days,
                                "dates": dates,
                                "values": values,
                                "obs_at_model": obs_at_model,
                                "diff_obs": values - obs_at_model,
                            }
                        )

            for ref in references:
                rows.append(
                    {
                        "variable": spec.model,
                        "series": ref["label"],
                        "run_id": "",
                        "description": spec.description,
                        "observation": ",".join(spec.obs_terms),
                        "units": spec.units,
                        **score(ref["values"], ref["obs_at_model"]),
                    }
                )
            for member in members:
                rows.append(
                    {
                        "variable": spec.model,
                        "series": "experiment",
                        "run_id": member["run_id"],
                        "description": spec.description,
                        "observation": ",".join(spec.obs_terms),
                        "units": spec.units,
                        **score(member["values"], member["obs_at_model"]),
                    }
                )

            data[spec.model] = {
                "spec": spec,
                "obs_days": obs_days,
                "obs_dates": obs_dates,
                "obs_values": obs_values,
                "references": references,
                "members": members,
            }

    summary = pd.DataFrame(rows)
    if not summary.empty:
        summary = summary.sort_values(["variable", "series", "rmse", "run_id"]).reset_index(drop=True)
    return data, summary


VARIABLE_SPECS = discover_variable_specs()
DATA, SUMMARY = load_comparison_batch(VARIABLE_SPECS, BATCH_FILES)
print(f"loaded {len(DATA)} mapped observation variables")
SUMMARY.head(20)


## Interactive Figure

In [ ]:
def variable_title(name: str, d: dict) -> str:
    spec = d["spec"]
    if d["members"]:
        run_ids = [m["run_id"] for m in d["members"]]
        run_text = f" | experiment runs {run_label(min(run_ids, key=sort_key))}-{run_label(max(run_ids, key=sort_key))}"
    else:
        run_text = ""
    return (
        f"{name}: observation, continuous baseline, stitched baseline"
        f"{', experiment' if d['members'] else ''}"
        f"<br><sup>{spec.description} | obs={','.join(spec.obs_terms)}{run_text}</sup>"
    )


def make_interactive_figure(data: dict[str, dict] = DATA) -> go.Figure:
    names = list(data)
    fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.10, row_heights=[0.68, 0.32])
    trace_counts = []

    for i, name in enumerate(names):
        d = data[name]
        spec = d["spec"]
        visible = i == 0
        start_count = len(fig.data)

        fig.add_trace(
            go.Scatter(
                x=d["obs_dates"],
                y=d["obs_values"],
                mode="lines",
                name="observation",
                line=dict(color="rgba(35,35,35,0.80)", width=1.4),
                visible=visible,
                hovertemplate="%{x}<br>obs=%{y:.4g}<extra></extra>",
            ),
            row=1,
            col=1,
        )

        for ref in d["references"]:
            fig.add_trace(
                go.Scatter(
                    x=ref["dates"],
                    y=ref["values"],
                    mode="lines",
                    name=ref["label"],
                    line=dict(color=ref["color"], width=ref["width"]),
                    visible=visible,
                    hovertemplate=f"%{{x}}<br>{ref['label']}=%{{y:.4g}}<extra></extra>",
                ),
                row=1,
                col=1,
            )
        for member in d["members"]:
            fig.add_trace(
                go.Scatter(
                    x=member["dates"],
                    y=member["values"],
                    mode="lines",
                    name=member["label"],
                    line=dict(width=1.0),
                    opacity=0.55,
                    visible=visible,
                    hovertemplate=f"%{{x}}<br>{member['label']}=%{{y:.4g}}<extra></extra>",
                ),
                row=1,
                col=1,
            )

        for ref in d["references"]:
            fig.add_trace(
                go.Scatter(
                    x=ref["dates"],
                    y=ref["diff_obs"],
                    mode="lines",
                    name=f"{ref['label']} - observation",
                    line=dict(color=ref["color"], width=1.6),
                    visible=visible,
                    hovertemplate=f"%{{x}}<br>{ref['label']} - obs=%{{y:.4g}}<extra></extra>",
                ),
                row=2,
                col=1,
            )
        for member in d["members"]:
            fig.add_trace(
                go.Scatter(
                    x=member["dates"],
                    y=member["diff_obs"],
                    mode="lines",
                    name=f"{member['label']} - observation",
                    line=dict(width=0.9),
                    opacity=0.45,
                    visible=visible,
                    hovertemplate=f"%{{x}}<br>{member['label']} - obs=%{{y:.4g}}<extra></extra>",
                ),
                row=2,
                col=1,
            )

        trace_counts.append(len(fig.data) - start_count)

    buttons = []
    offset = 0
    ranges = []
    for count in trace_counts:
        ranges.append((offset, offset + count))
        offset += count
    for idx, name in enumerate(names):
        visible = [False] * len(fig.data)
        start, stop = ranges[idx]
        for j in range(start, stop):
            visible[j] = True
        buttons.append(dict(label=name, method="update", args=[{"visible": visible}, {"title": variable_title(name, data[name])}]))

    first = data[names[0]]
    units = first["spec"].units
    fig.update_layout(
        title=variable_title(names[0], first),
        width=1100,
        height=760,
        hovermode="x unified",
        updatemenus=[dict(buttons=buttons, direction="down", x=1.02, xanchor="left", y=1.0, yanchor="top")],
        legend=dict(orientation="h", y=-0.14),
    )
    fig.update_yaxes(title_text=f"value ({units})" if units else "value", row=1, col=1)
    fig.update_yaxes(title_text=f"model - obs ({units})" if units else "model - obs", row=2, col=1, zeroline=True)
    fig.update_xaxes(title_text="Time", row=2, col=1)
    return fig


FIG = make_interactive_figure(DATA)
FIG


## Static Export

In [ ]:
def current_batch_label() -> str:
    if not EXPERIMENT_ENABLED:
        return "no_experiment"
    return "custom" if SELECTED_RUN_IDS is not None else f"batch{BATCH_INDEX:02d}"


def safe_name(name: str) -> str:
    return re.sub(r"[^A-Za-z0-9_.-]+", "_", name).strip("_")


def export_batch_pngs(data: dict[str, dict] = DATA, out_dir: Path = OUT_DIR) -> list[Path]:
    batch_label = current_batch_label()
    fig_dir = out_dir / batch_label
    fig_dir.mkdir(parents=True, exist_ok=True)
    paths = []

    for name, d in data.items():
        spec = d["spec"]
        fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True, gridspec_kw={"height_ratios": [2.1, 1.0], "hspace": 0.08})
        axes[0].plot(d["obs_dates"], d["obs_values"], color="0.20", lw=1.1, alpha=0.72, label="observation")
        for ref in d["references"]:
            axes[0].plot(ref["dates"], ref["values"], color=ref["color"], lw=ref["width"], label=ref["label"])
        for member in d["members"]:
            axes[0].plot(member["dates"], member["values"], lw=0.9, alpha=0.55, label=member["label"])
        axes[0].set_ylabel(f"{name} ({spec.units})" if spec.units else name)
        axes[0].legend(loc="best", frameon=False, ncols=2, fontsize=8)
        axes[0].grid(True, alpha=0.25)

        for ref in d["references"]:
            axes[1].plot(ref["dates"], ref["diff_obs"], color=ref["color"], lw=1.4, label=f"{ref['label']} - obs")
        for member in d["members"]:
            axes[1].plot(member["dates"], member["diff_obs"], lw=0.75, alpha=0.45)
        axes[1].axhline(0, color="black", lw=0.8)
        axes[1].set_ylabel(f"model - obs ({spec.units})" if spec.units else "model - obs")
        axes[1].set_xlabel("Time")
        axes[1].grid(True, alpha=0.25)

        if d["members"]:
            run_ids = [m["run_id"] for m in d["members"]]
            title_suffix = f", experiment runs {run_label(min(run_ids, key=sort_key))}-{run_label(max(run_ids, key=sort_key))}"
        else:
            title_suffix = ""
        fig.suptitle(
            f"{name}: observation, continuous baseline, stitched baseline{title_suffix}\n{spec.description} | obs={','.join(spec.obs_terms)}",
            x=0.02,
            ha="left",
        )
        fig.autofmt_xdate(rotation=0)
        fig.subplots_adjust(top=0.86, left=0.08, right=0.98, bottom=0.10)

        out = fig_dir / f"{safe_name(name)}_{batch_label}_observation_baseline_stitched_experiment.png"
        fig.savefig(out, dpi=160, bbox_inches="tight")
        plt.close(fig)
        paths.append(out)
    return paths


OUT_DIR.mkdir(parents=True, exist_ok=True)
summary_out = OUT_DIR / f"surface_summary_{current_batch_label()}.csv"
SUMMARY.to_csv(summary_out, index=False)
paths = export_batch_pngs(DATA)

print(summary_out)
print(f"exported {len(paths)} figures to {OUT_DIR / current_batch_label()}")
paths[:5]


## Optional: Export All Experiment Batches

In [ ]:
def export_all_batches(batch_size: int = BATCH_SIZE) -> pd.DataFrame:
    if not EXPERIMENT_ENABLED:
        raise ValueError("EXPERIMENT_ENABLED is False; there are no experiment batches to export")
    all_summaries = []
    n_batches = math.ceil(len(ALL_FILES) / batch_size)
    old_batch_index = globals().get("BATCH_INDEX")
    old_selected = globals().get("SELECTED_RUN_IDS")
    try:
        globals()["SELECTED_RUN_IDS"] = None
        for batch_index in range(n_batches):
            globals()["BATCH_INDEX"] = batch_index
            batch_files = select_batch(ALL_FILES)
            batch_data, batch_summary = load_comparison_batch(VARIABLE_SPECS, batch_files)
            export_batch_pngs(batch_data)
            batch_summary = batch_summary.copy()
            batch_summary.insert(0, "batch_index", batch_index)
            batch_summary.to_csv(OUT_DIR / f"surface_summary_batch{batch_index:02d}.csv", index=False)
            all_summaries.append(batch_summary)
            print(f"exported batch{batch_index:02d}: runs {[run_label(x) for x in batch_files['run_id']]}")
    finally:
        globals()["BATCH_INDEX"] = old_batch_index
        globals()["SELECTED_RUN_IDS"] = old_selected

    combined = pd.concat(all_summaries, ignore_index=True) if all_summaries else pd.DataFrame()
    out = OUT_DIR / "surface_summary_all_batches.csv"
    combined.to_csv(out, index=False)
    print(out)
    return combined


# ALL_SUMMARY = export_all_batches()
